In [ ]:
%pip install openmeteo-requests
%pip install requests-cache retry-requests

In [25]:
import openmeteo_requests
import polars as pl
import requests_cache
from retry_requests import retry
from datetime import datetime, timedelta, timezone

Functionise the below to make easier

In [ ]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Need to set this up so there can be multiple locations
_latitude = 52.52
_longitude = 13.41
# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"

# Would be better to define a dictionary of the variables then pass in the names/order to use in later arguments. Then there is no risk of assigning values of one and mislabelling them.
var_list = [
    "temperature_2m",
    "relative_humidity_2m"
]

params = {
	"latitude": _latitude,
	"longitude": _longitude,
	"hourly": var_list,
}
responses = openmeteo.weather_api(url, params=params)


# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(var_list.index("temperature_2m")).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(var_list.index("relative_humidity_2m")).ValuesAsNumpy()

# Setting up the datetime length
start = datetime.fromtimestamp(hourly.Time(), timezone.utc)
end = datetime.fromtimestamp(hourly.TimeEnd(), timezone.utc)
freq = timedelta(seconds = hourly.Interval())

# create the datafame 
hourly_dataframe_pl = pl.select(
    date = pl.datetime_range(start, end, freq, closed="left"),
    temperature_2m = hourly_temperature_2m,
    humidity = hourly_relative_humidity_2m,
    _lat = response.Latitude(),
    _long = response.Longitude()
)
print(hourly_dataframe_pl)

# to do:
# seperate function to process the data e.g. split out the date and time


Coordinates: 52.52000045776367°N 13.419998168945312°E
shape: (168, 5)
┌─────────────────────────┬────────────────┬──────────┬───────┬───────────┐
│ date                    ┆ temperature_2m ┆ humidity ┆ _lat  ┆ _long     │
│ ---                     ┆ ---            ┆ ---      ┆ ---   ┆ ---       │
│ datetime[μs, UTC]       ┆ f32            ┆ f32      ┆ f64   ┆ f64       │
╞═════════════════════════╪════════════════╪══════════╪═══════╪═══════════╡
│ 2026-03-19 00:00:00 UTC ┆ 7.0955         ┆ 68.0     ┆ 52.52 ┆ 13.419998 │
│ 2026-03-19 01:00:00 UTC ┆ 6.5455         ┆ 69.0     ┆ 52.52 ┆ 13.419998 │
│ 2026-03-19 02:00:00 UTC ┆ 5.8955         ┆ 71.0     ┆ 52.52 ┆ 13.419998 │
│ 2026-03-19 03:00:00 UTC ┆ 5.0455         ┆ 74.0     ┆ 52.52 ┆ 13.419998 │
│ 2026-03-19 04:00:00 UTC ┆ 4.7955         ┆ 74.0     ┆ 52.52 ┆ 13.419998 │
│ …                       ┆ …              ┆ …        ┆ …     ┆ …         │
│ 2026-03-25 19:00:00 UTC ┆ 13.0195        ┆ 50.0     ┆ 52.52 ┆ 13.419998 │
│ 2026-03-25 20:00

In [68]:
hourly.Variables(0).ValuesLength()

168